# Setup & Dependencies

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[0:5]:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_074_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_070_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_035_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_042_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/label/20240529_EO4_RES2_fl_pid_012_label.tif
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/pred.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/val.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/test.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/train.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/.ipynb_checkpoints/train-checkpoint.txt
/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/split/.ip

In [2]:
!pip install -q segmentation-models-pytorch albumentations rasterio timm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


# Imports & Configuration

In [3]:
import os, gc, math
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
import segmentation_models_pytorch as smp
import albumentations as A
from pathlib import Path
from sklearn.model_selection import KFold

# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT    = Path('/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data')
IMG_DIR      = DATA_ROOT / 'image'
LABEL_DIR    = DATA_ROOT / 'label'
SPLIT_DIR    = DATA_ROOT / 'split'
PRED_IMG_DIR = DATA_ROOT / 'prediction' / 'image'

OUTPUT_DIR = Path('/kaggle/working')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Hyperparameters ──────────────────────────────────────────────────────────
NUM_CLASSES     = 3
IN_CHANNELS     = 15                # Increased for AWEI inclusion
PATCH_SIZE      = 512
BATCH_SIZE      = 2                 
ACCUM_GRAD      = 4                 
NUM_EPOCHS      = 100               # SWA needs a longer tail to converge
LR              = 2e-4              
WEIGHT_DECAY    = 1e-3              # Increased to fight overfitting on 69 images
N_FOLDS         = 5
SEED            = 42
ENCODER         = 'efficientnet-b5' 
ENCODER_WEIGHTS = 'imagenet'

# OPTIMIZATION
USE_SWA         = True              
FLOOD_THRESHOLD = 0.38

# Dropout rate for decoder
DECODER_DROPOUT = 0.3

# Loss weights
TVERSKY_ALPHA   = 0.3      # FP penalty
TVERSKY_BETA    = 0.7      # FN penalty (flood recall)
LOVASZ_WEIGHT   = 0.5
TVERSKY_WEIGHT  = 0.5

# Class weights (estimated from training set: background ~80%, flood ~15%, water ~5%)
CLASS_WEIGHTS   = [0.2, 3.0, 1.5]

pl.seed_everything(SEED)
torch.backends.cudnn.benchmark = True
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"Encoder: {ENCODER}, Params: ~12M")

Seed set to 42


Device: CUDA
Encoder: efficientnet-b5, Params: ~12M


# Loading Splits

In [4]:
def load_ids(name):
    with open(SPLIT_DIR / f'{name}.txt') as f:
        return [l.strip() for l in f if l.strip()]

train_ids = load_ids('train')
val_ids   = load_ids('val')
test_ids  = load_ids('test')
pred_ids  = load_ids('pred')

ALL_TRAIN_IDS = train_ids + val_ids   # 69 images for 5-fold CV
PRED_IDS      = pred_ids

print(f"Train+Val (for CV): {len(ALL_TRAIN_IDS)}")
print(f"Test (held-out val): {len(test_ids)}")
print(f"Pred (final submission): {len(PRED_IDS)}")

Train+Val (for CV): 69
Test (held-out val): 10
Pred (final submission): 19


# Feature Engineering

| # | Channel | Source | Description |
|---|---------|--------|-------------|
| 1-2 | SAR_HH, SAR_HV | Raw | Log-scaled SAR backscatter |
| 3-6 | Green, Red, NIR, SWIR | Raw | Z-score normalised optical |
| 7 | NDWI | Derived | (Green−NIR)/(Green+NIR) |
| 8 | MNDWI | Derived | (Green−SWIR)/(Green+SWIR) |
| 9 | **Enhanced MNDWI** | Derived | tanh(3×MNDWI) — sharpens flood/water boundary |
| 10 | NDVI | Derived | (NIR−Red)/(NIR+Red) |
| 11 | SAR_ratio | Derived | log(HH/HV) — surface roughness |
| 12 | SAR_diff | Derived | log(HH)−log(HV) |
| 13 | DEM_proxy | Derived | −HV_log (low backscatter ≈ low elevation) |
| 14 | Water_acc | Derived | −MNDWI (water accumulation proxy) |


In [5]:
B_HH, B_HV, B_GREEN, B_RED, B_NIR, B_SWIR = 0, 1, 2, 3, 4, 5

# Per-band normalisation stats (computed from training set)
RAW_MEANS = np.array([841.13, 371.62, 1734.18, 1588.31, 1742.85, 1218.56], dtype=np.float32)
RAW_STDS  = np.array([473.71, 170.36,  623.05,  612.85,  564.58,  528.09], dtype=np.float32)


def load_raw_image(img_path):
    """Load 6-band GeoTIFF → float32 [6, H, W], NaN/Inf replaced with 0."""
    with rasterio.open(img_path) as src:
        raw = src.read().astype(np.float32)
    return np.where(np.isfinite(raw), raw, 0.0)


def engineer_15ch(raw):
    """Build 15-channel tensor. 
    Adds AWEI for shadow suppression and a SAR Mean Intensity band.
    """
    eps = 1e-6
    hh, hv       = raw[B_HH], raw[B_HV]
    green, red   = raw[B_GREEN], raw[B_RED]
    nir, swir    = raw[B_NIR], raw[B_SWIR]

    # Z-score normalization
    raw_norm = (raw - RAW_MEANS[:, None, None]) / (RAW_STDS[:, None, None] + eps)

    # Spectral Indices
    mndwi = np.clip((green - swir) / (green + swir + eps), -1.0, 1.0)
    ndvi  = np.clip((nir - red) / (nir + red + eps), -1.0, 1.0)
    
    # AWEI (sh) - Excellent for urban/mountainous shadow areas
    awei = 4.0 * (green - swir) - (0.25 * nir + 2.75 * red)
    awei = np.clip(awei / 5000.0, -1.0, 1.0) 

    # SAR Features
    hh_log = np.log1p(np.clip(hh, 0, None))
    hv_log = np.log1p(np.clip(hv, 0, None))
    sar_ratio = np.log(np.abs(hh) + eps) - np.log(np.abs(hv) + eps)

    stack = np.stack([
        raw_norm[B_HH], raw_norm[B_HV], 
        raw_norm[B_GREEN], raw_norm[B_RED],
        raw_norm[B_NIR], raw_norm[B_SWIR],
        mndwi, np.tanh(3.0 * mndwi),     
        ndvi, awei,                      # Channel 9: AWEI
        sar_ratio, hh_log - hv_log,      
        -hv_log, -mndwi,                 # Water proxies
        (hh_log + hv_log) / 2.0          # Channel 14: SAR Mean
    ], axis=0).astype(np.float32)

    return np.clip(stack, -10.0, 10.0)

# Sanity check
_t = engineer_15ch(np.random.rand(6, 512, 512).astype(np.float32) * 1000)
print(f"Feature shape: {_t.shape}  |  Range: [{_t.min():.2f}, {_t.max():.2f}]")

import gc
del _t
gc.collect()

Feature shape: (15, 512, 512)  |  Range: [-10.00, 10.00]


60

# Augmentation

In [6]:
TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15,
                       border_mode=0, p=0.4),
    A.OneOf([
        A.GaussNoise(var_limit=(0.001, 0.01), p=1.0),
        A.GaussianBlur(blur_limit=3, p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),
])

VAL_AUG = None   # No augmentation at test time (TTA handles it separately)

import random
from torch.utils.data import default_collate

def mixup_cutmix_collate(batch, mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5):
    images, masks = default_collate(batch)  # images: [B,C,H,W], masks: [B,H,W]
    if random.random() > prob:
        return images, masks

    use_cutmix = random.random() > 0.5
    lam = np.random.beta(mixup_alpha, mixup_alpha) if not use_cutmix else np.random.beta(cutmix_alpha, cutmix_alpha)

    batch_size = images.size(0)
    index = torch.randperm(batch_size)

    if use_cutmix:
        H, W = images.shape[2], images.shape[3]
        cx = np.random.randint(0, W)
        cy = np.random.randint(0, H)
        bw = int(np.round(W * np.sqrt(1 - lam)))
        bh = int(np.round(H * np.sqrt(1 - lam)))
        x1 = max(0, cx - bw // 2)
        x2 = min(W, cx + bw // 2)
        y1 = max(0, cy - bh // 2)
        y2 = min(H, cy + bh // 2)
        # Apply to images
        images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
        # Apply to masks - note the batch dimension
        masks[:, y1:y2, x1:x2] = masks[index, y1:y2, x1:x2]
        # Adjust lam to actual area
        lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    else:
        # MixUp
        images = lam * images + (1 - lam) * images[index]
        masks = lam * masks + (1 - lam) * masks[index]
        masks = masks.long()
    return images, masks

print("Augmentation pipelines ready.")

Augmentation pipelines ready.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_24/585516208.py:8: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(0.001, 0.01), p=1.0),
/tmp/ipykernel_24/585516208.py:11: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),


# DataSet

In [7]:
class FloodDataset(Dataset):
    """
    Loads 6-band GeoTIFFs, engineers 14-channel tensors, returns 3-class masks.
    Label convention:  0=No Flood  1=Flood  2=Water Body  -1=Ignore
    """
    def __init__(self, file_ids, img_dir, label_dir=None, transform=None):
        self.file_ids  = list(file_ids)
        self.img_dir   = Path(img_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.transform = transform

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        fid = self.file_ids[idx]

        # Image
        img_paths = list(self.img_dir.glob(f'*{fid}_image.tif'))
        assert img_paths, f"Image not found for {fid}"
        raw  = load_raw_image(img_paths[0])     # [6, H, W]
        feat = engineer_15ch(raw)               # [14, H, W]
        img_hw = feat.transpose(1, 2, 0)        # [H, W, 14] for albumentations

        # Label
        if self.label_dir is not None:
            lbl_paths = list(self.label_dir.glob(f'*{fid}_label.tif'))
            assert lbl_paths, f"Label not found for {fid}"
            with rasterio.open(lbl_paths[0]) as src:
                mask = src.read(1).astype(np.int64)
            mask = np.where((mask >= 0) & (mask <= 2), mask, -1)
        else:
            mask = np.zeros(feat.shape[1:], dtype=np.int64)

        # Augmentation
        if self.transform is not None:
            aug = self.transform(image=img_hw, mask=mask.astype(np.int32))
            img_hw = aug['image']
            mask   = aug['mask'].astype(np.int64)

        img_t  = torch.from_numpy(img_hw.transpose(2, 0, 1)).float()  # [14,H,W]
        mask_t = torch.from_numpy(mask).long()                         # [H,W]
        return img_t, mask_t


class PredDataset(Dataset):
    """Dataset for final prediction images (no labels)."""
    def __init__(self, file_ids, img_dir):
        self.file_ids = list(file_ids)
        self.img_dir  = Path(img_dir)

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        fid = self.file_ids[idx]
        img_paths = list(self.img_dir.glob(f'*{fid}_image.tif'))
        assert img_paths, f"Pred image not found for {fid}"
        raw  = load_raw_image(img_paths[0])
        feat = engineer_15ch(raw)
        return torch.from_numpy(feat).float(), fid

print("Dataset classes defined.")

Dataset classes defined.


# Loss

**Why Tversky over Dice?**
- Tversky(α=0.3, β=0.7): FN penalty = 2.3× FP penalty — critical for flood recall
- Per-class weights: Flood=80%, Water=15%, Background=5% — forces minority class focus
- Combined with Focal loss (γ=2, flood weight=10) for pixel-level calibration

In [8]:
# Option 2: Local implementation (copy this whole cell)
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from itertools import filterfalse as ifilterfalse

def lovasz_grad(gt_sorted):
    """Computes gradient of the Lovasz extension w.r.t sorted errors"""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    """Binary Lovasz hinge loss"""
    if len(labels) == 0:
        return logits.sum() * 0.
    signs = 2. * labels - 1.
    errors = 1. - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), grad)
    return loss

def lovasz_softmax_flat(probas, labels, classes='present'):
    """Multi-class Lovasz-Softmax loss"""
    if probas.numel() == 0:
        return probas * 0.
    C = probas.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if classes == 'present' and fg.sum() == 0:
            continue
        errors = (probas[:, c] - fg).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))
    return torch.stack(losses).mean()

def lovasz_softmax(logits, labels, ignore_index=-1, per_image=True):
    """Multi-class Lovasz-Softmax loss for semantic segmentation"""
    logits = logits.permute(0, 2, 3, 1).contiguous()
    labels = labels.contiguous()
    if per_image:
        def treat_image(log_i, lab_i):
            log_i = log_i.view(-1, log_i.size(-1))
            lab_i = lab_i.view(-1)
            mask = lab_i != ignore_index
            log_i = log_i[mask]
            lab_i = lab_i[mask]
            probas = F.softmax(log_i, dim=1)
            return lovasz_softmax_flat(probas, lab_i)
        losses = [treat_image(log_i, lab_i) for log_i, lab_i in zip(logits, labels)]
        loss = torch.stack(losses).mean()
    else:
        logits = logits.view(-1, logits.size(-1))
        labels = labels.view(-1)
        mask = labels != ignore_index
        logits = logits[mask]
        labels = labels[mask]
        probas = F.softmax(logits, dim=1)
        loss = lovasz_softmax_flat(probas, labels)
    return loss

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6, ignore_index=-1, class_weights=None):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.ignore_index = ignore_index
        self.class_weights = torch.tensor(class_weights) if class_weights else None

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        valid = (targets != self.ignore_index)
        tgt = targets.clone()
        tgt[~valid] = 0
        tgt_onehot = F.one_hot(tgt, num_classes=logits.size(1)).permute(0,3,1,2).float()
        probs = probs * valid.unsqueeze(1).float()

        loss = 0.0
        for c in range(logits.size(1)):
            p = probs[:, c]
            g = tgt_onehot[:, c]
            tp = (p * g).sum()
            fp = (p * (1 - g)).sum()
            fn = ((1 - p) * g).sum()
            tversky = (tp + self.smooth) / (tp + self.alpha*fp + self.beta*fn + self.smooth)
            class_weight = self.class_weights[c] if self.class_weights is not None else 1.0
            loss += class_weight * (1 - tversky)
        return loss

class CombinedLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=-1, lovasz_weight=0.5, tversky_weight=0.5,
                 tversky_alpha=0.3, tversky_beta=0.7, class_weights=None):
        super().__init__()
        self.lovasz_weight = lovasz_weight
        self.tversky_weight = tversky_weight
        self.tversky = TverskyLoss(alpha=tversky_alpha, beta=tversky_beta,
                                   ignore_index=ignore_index, class_weights=class_weights)

    def forward(self, logits, targets):
        # Use local lovasz_softmax function
        lovasz = lovasz_softmax(logits, targets, ignore_index=-1, per_image=True)
        tversky = self.tversky(logits, targets)
        return self.lovasz_weight * lovasz + self.tversky_weight * tversky

# UNet++ EfficientNet-B5

In [10]:
class FloodSegModel(pl.LightningModule):
    """
    UNet++ with EfficientNet-B5 encoder and scSE attention.
    14-channel input, 3-class output.
    Differential learning rates: encoder LR/10, decoder LR.
    """
    def __init__(self, in_channels=14, num_classes=3,
                 encoder_name='efficientnet-b3', encoder_weights='imagenet',
                 decoder_dropout=0.3, lr=3e-4, weight_decay=5e-4):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.num_classes = num_classes

        self.model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=num_classes,
            decoder_attention_type='scse',
            decoder_use_batchnorm=True,
            decoder_dropout=decoder_dropout,   # ← add dropout
        )
        self.criterion = CombinedLoss(
            num_classes=num_classes,
            ignore_index=-1,
            lovasz_weight=0.5,
            tversky_weight=0.5,
            tversky_alpha=0.3,
            tversky_beta=0.7,
            class_weights=CLASS_WEIGHTS
        )
        self._reset_iou('train')
        self._reset_iou('val')

    def _reset_iou(self, split):
        for attr in ['tp', 'fp', 'fn']:
            setattr(self, f'{split}_{attr}', torch.zeros(self.num_classes))

    def _update_iou(self, logits, masks, split):
        preds = torch.argmax(logits, dim=1)
        valid = masks != -1
        for c in range(self.num_classes):
            pc = (preds == c) & valid
            gc = (masks == c) & valid
            getattr(self, f'{split}_tp')[c] += (pc & gc).sum().float().cpu()
            getattr(self, f'{split}_fp')[c] += (pc & ~gc).sum().float().cpu()
            getattr(self, f'{split}_fn')[c] += (~pc & gc).sum().float().cpu()

    def _log_epoch_iou(self, split):
        tp   = getattr(self, f'{split}_tp')
        fp   = getattr(self, f'{split}_fp')
        fn   = getattr(self, f'{split}_fn')
        iou  = tp / (tp + fp + fn + 1e-6)
        miou = iou.mean()
        self.log(f'{split}_mIoU',      miou,   prog_bar=True,  sync_dist=True)
        self.log(f'{split}_IoU_BG',    iou[0], prog_bar=False, sync_dist=True)
        self.log(f'{split}_IoU_Flood', iou[1], prog_bar=True,  sync_dist=True)
        self.log(f'{split}_IoU_Water', iou[2], prog_bar=True,  sync_dist=True)
        self._reset_iou(split)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        imgs, masks = batch
        logits = self(imgs)
        loss   = self.criterion(logits, masks)
        self._update_iou(logits.detach(), masks, 'train')
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self._log_epoch_iou('train')

    def validation_step(self, batch, batch_idx):
        imgs, masks = batch
        logits = self(imgs)
        loss   = self.criterion(logits, masks)
        self._update_iou(logits.detach(), masks, 'val')
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        self._log_epoch_iou('val')

    def configure_optimizers(self):
        enc_params = list(self.model.encoder.parameters())
        dec_params = (list(self.model.decoder.parameters()) +
                      list(self.model.segmentation_head.parameters()))
        optimizer = torch.optim.AdamW([
            {'params': enc_params, 'lr': self.lr * 0.1},
            {'params': dec_params, 'lr': self.lr},
        ], weight_decay=self.hparams.weight_decay)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=1e-6)
        return {'optimizer': optimizer,
                'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}}

print("FloodSegModel defined.")


FloodSegModel defined.


# Cross Validation

In [11]:
# from pytorch_lightning.callbacks import StochasticWeightAveraging

# def train_fold(fold_idx, tr_ids, va_ids):
#     """Train one fold, return (best_ckpt_path, best_val_mIoU)."""
#     print(f"\n{'='*60}\n  FOLD {fold_idx+1}/{N_FOLDS}  |  Train={len(tr_ids)}  Val={len(va_ids)}\n{'='*60}")

#     train_ds = FloodDataset(tr_ids, IMG_DIR, LABEL_DIR, transform=TRAIN_AUG)
#     val_ds   = FloodDataset(va_ids, IMG_DIR, LABEL_DIR, transform=VAL_AUG)
#     train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
#                           num_workers=2, pin_memory=True, drop_last=True, collate_fn=mixup_cutmix_collate)
#     val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
#                           num_workers=2, pin_memory=True)

#     model = FloodSegModel(
#         in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
#         encoder_name=ENCODER, encoder_weights=ENCODER_WEIGHTS,
#         lr=LR, weight_decay=WEIGHT_DECAY,
#     )

#     ckpt_cb = ModelCheckpoint(
#         dirpath=CKPT_DIR,
#         filename=f'fold{fold_idx+1}-{{epoch:02d}}-{{val_mIoU:.4f}}',
#         monitor='val_mIoU', mode='max', save_top_k=1,
#     )
    
#     swa_cb = StochasticWeightAveraging(swa_lrs=1e-4, swa_epoch_start=0.7, annealing_epochs=5)   # start SWA after 70% of epochs
#     callbacks = [
#         ckpt_cb,
#         EarlyStopping(monitor='val_mIoU', mode='max', patience=15),
#         swa_cb
#     ]

#     trainer = pl.Trainer(
#         max_epochs=NUM_EPOCHS,
#         accelerator='auto',
#         devices=1,
#         precision='16-mixed',
#         accumulate_grad_batches=ACCUM_GRAD,   # ← gradient accumulation
#         gradient_clip_val=1.0,
#         log_every_n_steps=5,
#         callbacks=callbacks,
#         enable_model_summary=(fold_idx == 0),
#     )
#     trainer.fit(model, train_dl, val_dl)

#     best_path  = ckpt_cb.best_model_path
#     best_score = float(ckpt_cb.best_model_score)
#     print(f"  Fold {fold_idx+1} best val_mIoU: {best_score:.4f}  →  {best_path}")

#     del model, train_ds, val_ds, train_dl, val_dl, trainer
#     gc.collect(); torch.cuda.empty_cache()
#     return best_path, best_score


# kf        = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
# all_ids   = np.array(ALL_TRAIN_IDS)
# fold_ckpts = []    # list of (ckpt_path, val_miou)

# for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(all_ids)):
#     ckpt_path, val_score = train_fold(fold_idx,
#                                        all_ids[tr_idx].tolist(),
#                                        all_ids[va_idx].tolist())
#     fold_ckpts.append((ckpt_path, val_score))

# fold_scores = [s for _, s in fold_ckpts]
# print(f"\n{'='*60}\n5-Fold CV Summary:")
# for i, (p, s) in enumerate(fold_ckpts):
#     print(f"  Fold {i+1}: mIoU={s:.4f}  |  {Path(p).name}")
# print(f"  Mean mIoU: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
# print(f"{'='*60}")


# TTA & Ensemble

In [12]:
def tta_predict(model, images):
    aug_probs = []
    with torch.no_grad():
        # Original
        aug_probs.append(torch.softmax(model(images), dim=1))
        # Horizontal flip
        aug_probs.append(torch.softmax(model(torch.flip(images, [3])), dim=1).flip([3]))
        # Vertical flip
        aug_probs.append(torch.softmax(model(torch.flip(images, [2])), dim=1).flip([2]))
        # Rot90
        r90 = torch.rot90(images, 1, [2,3])
        aug_probs.append(torch.softmax(model(r90), dim=1).rot90(-1, [2,3]))
        # Rot270
        r270 = torch.rot90(images, 3, [2,3])
        aug_probs.append(torch.softmax(model(r270), dim=1).rot90(-3, [2,3]))
                
    return torch.stack(aug_probs).mean(0)

import cv2
from scipy import ndimage

def postprocess_flood_mask(mask: np.ndarray, min_area=10, fill_holes=True) -> np.ndarray:
    """
    mask: binary flood mask (0/1)
    Remove small connected components and optionally fill holes.
    """
    # Label connected components
    labeled, num_features = ndimage.label(mask)
    sizes = ndimage.sum(mask, labeled, range(num_features+1))
    # Keep only components larger than min_area
    for comp in range(1, num_features+1):
        if sizes[comp] < min_area:
            mask[labeled == comp] = 0
    # Fill small holes (optional)
    if fill_holes:
        mask = ndimage.binary_fill_holes(mask).astype(np.uint8)
    return mask

def run_ensemble_inference(ckpt_list, pred_ids, pred_img_dir, batch_size=2):
    """
    ckpt_list: [(path, val_score), ...]
    Returns: [(fid, pred_mask_np), ...] where pred_mask_np is [H,W] with values 0/1/2
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pred_ds = PredDataset(pred_ids, pred_img_dir)
    pred_dl = DataLoader(pred_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    # Softmax ensemble weights from val scores
    scores  = np.array([s for _, s in ckpt_list], dtype=np.float32)
    weights = np.exp(scores) / np.exp(scores).sum()
    print(f"Ensemble: {len(ckpt_list)} models | weights: {[f'{w:.3f}' for w in weights]}")

    # Accumulate weighted probability maps (one entry per sample)
    cum_probs = None   # list of [C, H, W] tensors

    for model_i, ((ckpt_path, _), w) in enumerate(zip(ckpt_list, weights)):
        print(f"  [{model_i+1}/{len(ckpt_list)}] {Path(ckpt_path).name}")
        m = FloodSegModel.load_from_checkpoint(ckpt_path).to(device)
        m.eval()

        model_probs = []
        for imgs, _ in pred_dl:
            imgs = imgs.to(device)
            p    = tta_predict(m, imgs)   # [B, C, H, W]
            model_probs.extend(p.cpu().unbind(0))

        if cum_probs is None:
            cum_probs = [w * p for p in model_probs]
        else:
            for j, p in enumerate(model_probs):
                cum_probs[j] += w * p

        del m, model_probs
        gc.collect(); torch.cuda.empty_cache()
        print(f"    done.")

    # Collect fids in order
    fid_list = []
    for _, fids in DataLoader(pred_ds, batch_size=1, shuffle=False):
        fid_list.extend(fids)

    results = []
    for fid, prob_map in zip(fid_list, cum_probs):
        # prob_map: [3, H, W]
        # Instead of argmax, we manually define the classes
        
        # 1. Start with Background (Class 0)
        final_mask = np.zeros((512, 512), dtype=np.uint8)
        
        # 2. Assign Water Body (Class 2) if prob is high
        water_mask = (prob_map[2] > 0.5).numpy()
        final_mask[water_mask] = 2
        
        # 3. Assign Flood (Class 1) using the OPTIMIZED threshold
        # This usually provides a massive IoU boost on the LB
        flood_mask = (prob_map[1] > FLOOD_THRESHOLD).numpy()
        final_mask[flood_mask] = 1
        
        # Post-process the flood mask specifically
        cleaned_flood = (final_mask == 1).astype(np.uint8)
        cleaned_flood = postprocess_flood_mask(cleaned_flood, min_area=120)
        
        # Re-apply cleaned flood to final mask
        final_mask[final_mask == 1] = 0
        final_mask[cleaned_flood == 1] = 1
        
        results.append((fid, final_mask))

    return results

# # ── Build ensemble checkpoint list ───────────────────────────────────────────
# ensemble_ckpts = list(fold_ckpts)                        # all 5 folds

# # Drop weak models (val_mIoU < 0.25) to avoid ensemble degradation
# ensemble_ckpts = [(p, s) for p, s in ensemble_ckpts if s > 0.25]
# if not ensemble_ckpts:
#     print("WARNING: No models cleared quality filter — using all 5 folds.")
#     ensemble_ckpts = list(fold_ckpts)

# print(f"\nRunning ensemble on {len(PRED_IDS)} test patches ...")
# predictions = run_ensemble_inference(ensemble_ckpts, PRED_IDS, PRED_IMG_DIR, batch_size=2)
# print(f"Inference done. {len(predictions)} predictions.")


# Submission

In [13]:
def mask_to_rle(mask: np.ndarray) -> str:
    """
    Binary flood mask → RLE string (column-major, 1-indexed).
    Returns '0 0' if no flood pixels (as per competition rules).
    """
    pixels = mask.flatten(order='F')          # column-major (Kaggle convention)
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    rle_str = ' '.join(str(x) for x in runs)
    return rle_str if rle_str else '0 0'


# submission_rows    = []
# flood_pixel_counts = []

# for fid, pred_mask in predictions:
#     flood_mask = (pred_mask == 1).astype(np.uint8)   # Class 1 = Flood
#     flood_px   = int(flood_mask.sum())
#     flood_pixel_counts.append(flood_px)
#     submission_rows.append({'id': fid, 'rle_mask': mask_to_rle(flood_mask)})

# df_sub = pd.DataFrame(submission_rows)
# df_sub['rle_mask'] = df_sub['rle_mask'].replace('', '0 0').fillna('0 0')

# sub_path = str(OUTPUT_DIR / 'submission_v5_ensemble_tta.csv')
# df_sub.to_csv(sub_path, index=False)

# print(f"Submission saved: {sub_path}")
# print(f"Total rows:       {len(df_sub)}")
# print(f"Flood pixels — min={min(flood_pixel_counts)}  max={max(flood_pixel_counts)}  "
#       f"mean={np.mean(flood_pixel_counts):.0f}")
# print(f"\nSample rows:")
# print(df_sub.head(5).to_string(index=False))

# Local Validation

In [14]:
# # Estimate mIoU on held-out test split using the ensemble
# # Helps you gauge your Kaggle score before submitting.
# print("Computing local mIoU on test split ...")

# device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# test_ds   = FloodDataset(test_ids, IMG_DIR, LABEL_DIR, transform=VAL_AUG)
# test_dl   = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=2)

# scores_arr  = np.array([s for _, s in ensemble_ckpts], dtype=np.float32)
# weights_arr = np.exp(scores_arr) / np.exp(scores_arr).sum()

# tp_acc = torch.zeros(NUM_CLASSES)
# fp_acc = torch.zeros(NUM_CLASSES)
# fn_acc = torch.zeros(NUM_CLASSES)

# for imgs, masks in test_dl:
#     imgs  = imgs.to(device)
#     batch_probs = None

#     for (ckpt_path, _), w in zip(ensemble_ckpts, weights_arr):
#         m = FloodSegModel.load_from_checkpoint(ckpt_path).to(device)
#         m.eval()
#         with torch.no_grad():
#             probs = tta_predict(m, imgs)
#         batch_probs = w * probs if batch_probs is None else batch_probs + w * probs
#         del m; torch.cuda.empty_cache()

#     preds = torch.argmax(batch_probs.cpu(), dim=1)
#     valid = masks != -1
#     for c in range(NUM_CLASSES):
#         pc = (preds == c) & valid
#         gc = (masks == c) & valid
#         tp_acc[c] += (pc & gc).sum().float()
#         fp_acc[c] += (pc & ~gc).sum().float()
#         fn_acc[c] += (~pc & gc).sum().float()

# iou  = tp_acc / (tp_acc + fp_acc + fn_acc + 1e-6)
# miou = iou.mean()

# print(f"\n{'='*50}")
# print(f"  LOCAL VALIDATION (test split)")
# print(f"{'='*50}")
# print(f"  No Flood IoU   : {iou[0]:.4f}")
# print(f"  Flood IoU      : {iou[1]:.4f}")
# print(f"  Water Body IoU : {iou[2]:.4f}")
# print(f"  mIoU           : {miou:.4f}")
# print(f"{'='*50}")

# # Flood recall diagnostic
# flood_tp = tp_acc[1].item()
# flood_fn = fn_acc[1].item()
# flood_recall = flood_tp / (flood_tp + flood_fn + 1e-6)
# print(f"  Flood Recall   : {flood_recall:.4f}  (% of actual flood pixels found)")

# Kaggle Hub Upload

In [15]:
# import kagglehub
# import shutil

# # ── Configuration — edit these ────────────────────────────────────────────────
# KAGGLE_USERNAME = "devamwppu2000828"   # e.g. "tjhansichandrareddy"
# MODEL_SLUG      = "aisehack-flood-unetpp-efficientnet-b5"      # must be lowercase, hyphens only
# FRAMEWORK       = "pytorch"
# VARIATION       = "v3"
# VERSION_NOTES   = (
#     "UNet++ EfficientNet-B5 | 14-channel feature engineering | "
#     "Tversky+Focal loss | 5-fold CV ensemble | 8-way TTA"
# )

# HANDLE = f"{KAGGLE_USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}"

# # ── Stage all checkpoints into a single export directory ─────────────────────
# EXPORT_DIR = OUTPUT_DIR / "model_export"
# EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# # Copy all fold checkpoints
# for ckpt_path, score in fold_ckpts:
#     shutil.copy(ckpt_path, EXPORT_DIR / Path(ckpt_path).name)
#     print(f"  Staged: {Path(ckpt_path).name}  (val_mIoU={score:.4f})")

# # Save metadata alongside checkpoints
# meta = {
#     "fold_checkpoints": [{"path": Path(p).name, "val_miou": s} for p, s in fold_ckpts],
#     "ensemble_checkpoints": [{"path": Path(p).name, "val_miou": s} for p, s in ensemble_ckpts],
#     "config": {
#         "in_channels": IN_CHANNELS,
#         "num_classes": NUM_CLASSES,
#         "encoder": ENCODER,
#         "encoder_weights": ENCODER_WEIGHTS,
#         "patch_size": PATCH_SIZE,
#     }
# }
# import json as _json
# with open(EXPORT_DIR / "metadata.json", "w") as f:
#     _json.dump(meta, f, indent=2)
# print(f"  Staged: metadata.json")

# staged_files = list(EXPORT_DIR.iterdir())
# print(f"\nTotal files staged: {len(staged_files)}")
# for sf in staged_files:
#     size_mb = sf.stat().st_size / 1e6
#     print(f"  {sf.name:<60}  {size_mb:.1f} MB")

# # ── Upload to Kaggle Hub ───────────────────────────────────────────────────────
# print(f"\nUploading to: {HANDLE}")
# try:
#     kagglehub.model_upload(
#         handle=HANDLE,
#         local_model_dir=str(EXPORT_DIR),
#         version_notes=VERSION_NOTES,
#     )
#     print(f"Upload successful!")
#     print(f"Model available at: https://www.kaggle.com/models/{HANDLE}")
# except Exception as e:
#     print(f"Upload failed: {e}")
#     print("Tip: Make sure kaggle.json is configured and the model slug exists on Kaggle.")


In [16]:
# import time
# time.sleep(15)

In [17]:
# import kagglehub, json as _json, re
# from pathlib import Path

# # ── Edit this to match your upload handle ────────────────────────────────────
# HANDLE = "devamwppu2000828/aisehack-flood-unetpp-efficientnet-b5/pytorch/v3"

# print(f"Downloading model: {HANDLE}")
# model_dir = Path(kagglehub.model_download(HANDLE))
# print(f"Downloaded to: {model_dir}")

# # ── List available files ─────────────────────────────────────────────────────
# ckpt_files = sorted(model_dir.glob("*.ckpt"))
# print(f"\nFound {len(ckpt_files)} checkpoint(s):")
# for f in ckpt_files:
#     print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

# # ── Load metadata if available ───────────────────────────────────────────────
# meta_path = model_dir / "metadata.json"
# if meta_path.exists():
#     with open(meta_path) as f:
#         meta = _json.load(f)
#     print(f"\nMetadata loaded:")
#     print(f"  Encoder:     {meta['config']['encoder']}")
#     print(f"  In channels: {meta['config']['in_channels']}")
#     print(f"  Folds:       {len(meta['fold_checkpoints'])}")
#     ensemble_meta = meta["ensemble_checkpoints"]
# else:
#     # Fallback: parse val_mIoU from filenames
#     def extract_miou(p):
#         m = re.search(r'val_mIoU[=_]?([0-9]+\.[0-9]+)', p.name)
#         return float(m.group(1)) if m else 0.0
#     ensemble_meta = [{"path": f.name, "val_miou": extract_miou(f)} for f in ckpt_files]
#     meta = {"config": {"in_channels": 14, "num_classes": 3,
#                         "encoder": "efficientnet-b5", "encoder_weights": "imagenet"}}
#     print("No metadata.json found — inferring from filenames.")

# # ── Reconstruct ensemble_ckpts list (path, score) ────────────────────────────
# downloaded_ckpts = [
#     (str(model_dir / e["path"]), e["val_miou"])
#     for e in ensemble_meta
#     if (model_dir / e["path"]).exists()
# ]
# downloaded_ckpts.sort(key=lambda x: x[1], reverse=True)  # best first

# print(f"\nEnsemble checkpoints ({len(downloaded_ckpts)} models):")
# for p, s in downloaded_ckpts:
#     print(f"  val_mIoU={s:.4f}  |  {Path(p).name}")

# # Update config from metadata
# IN_CHANNELS     = meta["config"]["in_channels"]
# NUM_CLASSES     = meta["config"]["num_classes"]
# ENCODER         = meta["config"]["encoder"]
# ENCODER_WEIGHTS = meta["config"]["encoder_weights"]
# print(f"\nConfig restored — IN_CHANNELS={IN_CHANNELS}  ENCODER={ENCODER}")


In [18]:
import kagglehub
import json

HANDLE = "devamwppu2000828/aisehack-flood-unetpp-efficientnet-b5/pytorch/v3"
print(f"Downloading model from: {HANDLE}")
model_dir = Path(kagglehub.model_download(HANDLE))
print(f"Model downloaded to: {model_dir}")

# Load metadata
meta_path = model_dir / "metadata.json"
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    encoder_name = meta["config"]["encoder"]
    in_channels  = meta["config"]["in_channels"]
    num_classes  = meta["config"]["num_classes"]
    # Get list of checkpoint files from metadata
    ckpt_files = [model_dir / e["path"] for e in meta["ensemble_checkpoints"]]
else:
    # Fallback: use all .ckpt files in directory
    encoder_name = "efficientnet-b5"   # default
    in_channels = 14
    num_classes = 3

ckpt_files = sorted(model_dir.glob("*.ckpt"))

print(f"Found {len(ckpt_files)} checkpoint(s):")
for f in ckpt_files:
    print(f"  {f.name}")

def predict_flood(image_paths, ckpt_paths, batch_size=1, min_area=50):
    """
    image_paths: list of paths to 6‑band GeoTIFFs
    ckpt_paths: list of checkpoint paths (ensemble)
    Returns: list of (image_id, flood_mask_np) where flood_mask is binary uint8
    """
    # Load all models
    models = []
    for cp in ckpt_paths:
        model = FloodSegModel(encoder_name=encoder_name, in_channels=in_channels, num_classes=num_classes)
        
        # Get the state dict from the checkpoint
        state = torch.load(cp, map_location='cpu')['state_dict']
        model.load_state_dict(state) 
        
        model.to('cuda')
        model.eval()
        models.append(model)

    results = []
    for img_path in image_paths:
        # Load and preprocess
        raw = load_raw_image(img_path)
        feat = engineer_15ch(raw)           # [14, H, W]
        img_tensor = torch.from_numpy(feat).unsqueeze(0).to('cuda')  # [1,14,H,W]

        # Ensemble TTA
        ensemble_prob = None
        for model in models:
            prob = tta_predict(model, img_tensor)   # [1,3,H,W]
            if ensemble_prob is None:
                ensemble_prob = prob
            else:
                ensemble_prob += prob
        ensemble_prob /= len(models)

        # Get flood class (class index 1)
        flood_prob = ensemble_prob[0, 1].cpu().numpy() 

        # 2. Apply the manual threshold
        # Using a value between 0.3 and 0.45 often yields better IoU than argmax
        flood_mask = (flood_prob > FLOOD_THRESHOLD).astype(np.uint8)
        
        # pred = torch.argmax(ensemble_prob, dim=1).squeeze(0).cpu().numpy()  # [H,W]
        # flood_mask = (pred == 1).astype(np.uint8)
        
        flood_mask = postprocess_flood_mask(flood_mask, min_area=min_area, fill_holes=True)

        # Extract image ID from filename
        img_id = Path(img_path).stem.replace('_image', '')
        results.append((img_id, flood_mask))

    # Cleanup
    for m in models:
        del m
    torch.cuda.empty_cache()
    return results

test_image_folder = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/prediction/image"   # change this
image_paths = list(Path(test_image_folder).glob("*_image.tif"))
print(f"Found {len(image_paths)} images")

# Use all downloaded checkpoints for ensemble
ckpt_paths = [str(p) for p in ckpt_files]

predictions = predict_flood(image_paths, ckpt_paths, batch_size=1, min_area=50)

# Convert to RLE submission
def mask_to_rle(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    rle_str = ' '.join(str(x) for x in runs)
    return rle_str if rle_str else '0 0'

sub_rows = []
for img_id, flood_mask in predictions:
    sub_rows.append({'id': img_id, 'rle_mask': mask_to_rle(flood_mask)})

df = pd.DataFrame(sub_rows)
df.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Model downloaded to: /kaggle/input/models/devamwppu2000828/aisehack-flood-unetpp-efficientnet-b5/pytorch/v3/1
Found 5 checkpoint(s):
  fold1-epoch50-val_mIoU0.6128.ckpt
  fold2-epoch17-val_mIoU0.5716.ckpt
  fold3-epoch20-val_mIoU0.6278.ckpt
  fold4-epoch18-val_mIoU0.5746.ckpt
  fold5-epoch43-val_mIoU0.6048.ckpt
Found 19 images


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

Submission saved to submission.csv


In [19]:
df.head

<bound method NDFrame.head of                               id  \
0   20240529_EO4_RES2_fl_pid_089   
1   20240529_EO4_RES2_fl_pid_095   
2   20240529_EO4_RES2_fl_pid_084   
3   20240529_EO4_RES2_fl_pid_097   
4   20240529_EO4_RES2_fl_pid_083   
5   20240529_EO4_RES2_fl_pid_094   
6   20240529_EO4_RES2_fl_pid_092   
7   20240529_EO4_RES2_fl_pid_080   
8   20240529_EO4_RES2_fl_pid_091   
9   20240529_EO4_RES2_fl_pid_098   
10  20240529_EO4_RES2_fl_pid_087   
11  20240529_EO4_RES2_fl_pid_090   
12  20240529_EO4_RES2_fl_pid_093   
13  20240529_EO4_RES2_fl_pid_086   
14  20240529_EO4_RES2_fl_pid_085   
15  20240529_EO4_RES2_fl_pid_096   
16  20240529_EO4_RES2_fl_pid_081   
17  20240529_EO4_RES2_fl_pid_082   
18  20240529_EO4_RES2_fl_pid_088   

                                             rle_mask  
0   32769 5 33281 7 33793 8 34305 9 34817 10 35329...  
1   11476 4 11987 7 12498 8 13010 9 13522 9 14034 ...  
2   114 39 240 14 294 72 378 19 626 13 643 21 754 ...  
3   12060 4 12571 7 13083